# NB02 · Building 3D Structures: Cartesian, Internal Coordinates and Editors
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prateekpaul/Cheminformatics/blob/main/solutions/nb02_building_3d_structures.ipynb)

**Estimated time:** ~60 min | **Difficulty:** ●●○

A comprehensive practical guide on constructing 3D molecular structures from geometric principles, distinguishing coordinate-dependent Cartesian frames from invariant internal coordinates (Z-matrix), implementing trigonometric conversions (NeRF), visualizing 3D conformations with `py3Dmol`, and performing torsional scans.

## Learning Objectives

By the end of this notebook, you will be able to:
1. **Construct** 3D Cartesian coordinates of methane from fundamental bond lengths and tetrahedral bond angles using trigonometry.
2. **Demonstrate** that Cartesian coordinates depend on the reference frame whereas interatomic distance matrices are invariant under rigid 3D rotation.
3. **Parse and formulate** internal coordinate representations (Z-matrices) for simple and multi-rotor molecules (methane, n-butane, 1,2-dichloroethane).
4. **Implement** an internal-to-Cartesian coordinate converter based on the Natural Extension Reference Frame (NeRF) formulation.
5. **Visualize** 3D molecular conformations using `py3Dmol` and compute torsional energy profiles (gauche vs. anti minima).
6. **Survey** common 2D and 3D molecular editors and sketchers used in computational chemistry.

## Prerequisites and Links

- NB00 and NB01 completed.
- Lecture 01 (Building 3D structures, trigonometry, Z-matrix, Exercise-1) and Lecture 02 (Cartesian coordinates, internal coordinates).

In [ ]:
import os, random, warnings, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem

SEED = 42
FULL_RUN = False
random.seed(SEED)
np.random.seed(SEED)
RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore", category=DeprecationWarning)

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
print("NB02 setup completed.")

## 1. Methane Construction by Trigonometry

In a perfect tetrahedral geometry ($sp^3$ hybridization), the bond angle is:

$$\alpha = \arccos\left(-\frac{1}{3}\right) \approx 109.4712^\circ \approx 1.91063\ \text{radians}$$

Let carbon be situated at the origin $(0, 0, 0)$. Defining the half-angle:

$$\theta = \frac{\alpha}{2} \approx 54.7356^\circ \approx 0.95532\ \text{radians}$$

With a standard C–H bond length $r = 1.09\ \text{Å}$, the four hydrogen positions are parameterized as:

$$\text{H}_1 = (r \cos\theta,\ r \sin\theta,\ 0)$$
$$\text{H}_2 = (r \cos\theta,\ -r \sin\theta,\ 0)$$
$$\text{H}_3 = (-r \cos\theta,\ 0,\ r \sin\theta)$$
$$\text{H}_4 = (-r \cos\theta,\ 0,\ -r \sin\theta)$$

Notice the numerical values:
- $r \cos\theta = 1.09 \times \cos(54.7356^\circ) = 1.09 \times 0.57735 \approx 0.6293\ \text{Å} \approx 0.63\ \text{Å}$
- $r \sin\theta = 1.09 \times \sin(54.7356^\circ) = 1.09 \times 0.81650 \approx 0.8900\ \text{Å} \approx 0.89\ \text{Å}$

> [!NOTE] What to notice
> The classic coordinates on the course slide $(\pm 0.63, \pm 0.89, 0)$ correspond exactly to these trigonometric projections for $r = 1.09\ \text{Å}$!

In [ ]:
r = 1.09  # Angstroms
alpha = np.arccos(-1.0 / 3.0)  # ~109.4712 degrees
theta = alpha / 2.0           # ~54.7356 degrees

# Calculate positions
C = np.array([0.0, 0.0, 0.0])
H1 = np.array([ r * np.cos(theta),  r * np.sin(theta), 0.0])
H2 = np.array([ r * np.cos(theta), -r * np.sin(theta), 0.0])
H3 = np.array([-r * np.cos(theta), 0.0,  r * np.sin(theta)])
H4 = np.array([-r * np.cos(theta), 0.0, -r * np.sin(theta)])

coords_methane = np.array([C, H1, H2, H3, H4])
atom_names = ["C", "H1", "H2", "H3", "H4"]

print("Methane Cartesian Coordinates (Angstroms):")
for name, c in zip(atom_names, coords_methane):
    print(f"{name:2s}: ({c[0]:7.4f}, {c[1]:7.4f}, {c[2]:7.4f})")

# Verify all 4 C-H bond lengths
ch_lengths = [np.linalg.norm(coords_methane[i] - C) for i in range(1, 5)]
print(f"C-H Bond lengths: {[round(l, 4) for l in ch_lengths]}")
assert all(np.isclose(l, 1.09, atol=1e-3) for l in ch_lengths)

# Verify H-C-H bond angles
def calc_angle(p1, center, p2):
    v1 = p1 - center
    v2 = p2 - center
    cos_val = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
    return np.degrees(np.arccos(np.clip(cos_val, -1.0, 1.0)))

angles = [
    calc_angle(H1, C, H2),
    calc_angle(H1, C, H3),
    calc_angle(H1, C, H4),
    calc_angle(H2, C, H3),
    calc_angle(H2, C, H4),
    calc_angle(H3, C, H4)
]
print(f"H-C-H Bond angles (degrees): {[round(a, 2) for a in angles]}")
assert all(np.isclose(a, 109.47, atol=1e-1) for a in angles)

## 2. Orientation Dependence vs. Distance Invariance

If we apply a rigid 3D rotation to the molecular coordinates:

$$X' = X R$$

where $R \in SO(3)$ is an orthogonal rotation matrix ($R R^T = I$), the Cartesian coordinates change dramatically.

However, the **interatomic Euclidean distance matrix**:

$$D_{ij} = \|x_i - x_j\|$$

is strictly invariant under any rigid translation or rotation.

> [!NOTE] Are molecular properties coordinate-dependent?
> As raised in the lecture slides: **Intrinsic scalar properties** (molecular energy, dipole moment magnitude, molecular volume, vibrational frequencies) are coordinate-invariant. **Directional properties** (dipole vector components along laboratory axes, quadrupole tensors) depend on the choice of external coordinate frame.

In [ ]:
# Define arbitrary 3D rotation matrix around axis
def rotation_matrix_3d(axis, angle_rad):
    axis = axis / np.linalg.norm(axis)
    a = np.cos(angle_rad / 2.0)
    b, c, d = -axis * np.sin(angle_rad / 2.0)
    return np.array([
        [a*a + b*b - c*c - d*d, 2*(b*c - a*d),     2*(b*d + a*c)],
        [2*(b*c + a*d),     a*a + c*c - b*b - d*d, 2*(c*d - a*b)],
        [2*(b*d - a*c),     2*(c*d + a*b),     a*a + d*d - b*b - c*c]
    ])

R_rot = rotation_matrix_3d(np.array([1.0, 1.0, 0.5]), np.radians(45))
coords_rotated = coords_methane @ R_rot.T

print("Coordinates changed under rotation:")
print(f"Original H1: {np.round(coords_methane[1], 4)}")
print(f"Rotated  H1: {np.round(coords_rotated[1], 4)}")

# Compute pairwise distance matrices
D_orig = np.linalg.norm(coords_methane[:, None, :] - coords_methane[None, :, :], axis=-1)
D_rot  = np.linalg.norm(coords_rotated[:, None, :] - coords_rotated[None, :, :], axis=-1)

assert np.allclose(D_orig, D_rot, atol=1e-5)
print("✔ Interatomic distance matrix is strictly invariant under rotation!")

## 3. Internal Coordinates: The Z-Matrix

Cartesian coordinates require $3N$ values and specify both molecular geometry and absolute orientation in space. For an isolated molecule, 6 degrees of freedom (3 translational, 3 rotational) are chemically irrelevant.

An **internal coordinate representation** specifies geometry using:
1. **Bond lengths** ($r_{ij}$, distance between atom $i$ and atom $j$).
2. **Bond angles** ($\theta_{ijk}$, angle formed by atoms $i-j-k$).
3. **Dihedral / Torsion angles** ($\phi_{ijkl}$, dihedral angle between planes $ijk$ and $jkl$).

The standard format is the **Z-Matrix**:
- Atom 1: Placed at the origin.
- Atom 2: Placed on the z-axis at distance $r_{21}$ from Atom 1.
- Atom 3: Placed in the xz-plane at distance $r_{32}$ from Atom 2 and angle $\theta_{321}$.
- Atom $i$ ($i \ge 4$): Placed relative to three previously defined atoms $j, k, l$ with length $r_{ij}$, angle $\theta_{ijk}$, and dihedral $\phi_{ijkl}$.

### Examples of Z-Matrices

**1. Methane ($\text{CH}_4$):**
```text
C
H  1  1.09
H  1  1.09  2  109.47
H  1  1.09  2  109.47  3  120.0
H  1  1.09  2  109.47  3  240.0
```

**2. n-Butane Anti Conformer ($\text{C}_4\text{H}_{10}$ heavy skeleton):**
```text
C1
C2  1  1.54
C3  2  1.54  1  109.5
C4  3  1.54  2  109.5  1  180.0
```

## 4. Z-Matrix to Cartesian Conversion (NeRF)

The **Natural Extension Reference Frame (NeRF)** algorithm efficiently converts internal coordinates into Cartesian space.

Given three existing Cartesian points $A, B, C$, to place atom $D$ with bond length $r = \|D - C\|$, bond angle $\theta = \angle BCD$, and dihedral angle $\phi = \angle ABCD$:
1. Construct the unit vector along the bond $C \to B$:
   $$u = \frac{C - B}{\|C - B\|}$$
2. Construct the normal vector to the plane $ABC$:
   $$w = \frac{u \times (B - A)}{\|u \times (B - A)\|}$$
3. Construct the orthogonal unit vector in the plane:
   $$v = w \times u$$
4. Atom $D$ is placed at:
   $$D = C + r \left( -\cos(\theta) u + \sin(\theta)\cos(\phi) v + \sin(\theta)\sin(\phi) w \right)$$

In [ ]:
def place_atom_nerf(p_a, p_b, p_c, r, angle_deg, dihedral_deg):
    """Place atom D given reference atoms A, B, C using NeRF method."""
    theta = np.radians(angle_deg)
    phi = np.radians(dihedral_deg)
    
    bc = p_c - p_b
    u = bc / np.linalg.norm(bc)
    
    ab = p_b - p_a
    cross_ab_bc = np.cross(u, ab)
    norm_cross = np.linalg.norm(cross_ab_bc)
    if norm_cross < 1e-7:  # Collinear points fallback
        w = np.array([0.0, 0.0, 1.0])
    else:
        w = cross_ab_bc / norm_cross
        
    v = np.cross(w, u)
    
    d_dir = -np.cos(theta) * u + np.sin(theta) * np.cos(phi) * v + np.sin(theta) * np.sin(phi) * w
    return p_c + r * d_dir

# Test: Place 4 heavy atoms of n-butane (anti conformer)
C1 = np.array([0.0, 0.0, 0.0])
C2 = np.array([1.54, 0.0, 0.0])
# Place C3 with r=1.54, angle=109.5 in XY plane
theta_c = np.radians(109.5)
C3 = C2 + np.array([-1.54 * np.cos(theta_c), 1.54 * np.sin(theta_c), 0.0])
# Place C4 with dihedral=180.0
C4 = place_atom_nerf(C1, C2, C3, 1.54, 109.5, 180.0)

coords_butane_anti = np.array([C1, C2, C3, C4])
print("Built Butane Anti-Conformer Heavy Atoms:")
for i, c in enumerate(coords_butane_anti, 1):
    print(f"C{i}: {np.round(c, 4)}")

## 5. Interactive 3D Visualization and Butane Torsion Scan

Conformational search explores rotation around single bonds. For n-butane ($CH_3-CH_2-CH_2-CH_3$), scanning the central C2–C3 dihedral angle $\phi$ from $0^\circ$ to $360^\circ$ produces the classic torsional energy profile:

- **Anti-periplanar ($\phi = 180^\circ$):** Global minimum ($0$ kJ/mol, steric repulsion minimized).
- **Gauche ($\phi \approx 60^\circ, 300^\circ$):** Local minima ($\approx 3.8$ kJ/mol above anti).
- **Anticlinal (eclipsed, $\phi \approx 120^\circ, 240^\circ$):** Energy barriers ($\approx 16$ kJ/mol).
- **Syn-periplanar (fully eclipsed, $\phi = 0^\circ$):** Maximum barrier ($\approx 19$ kJ/mol, methyl-methyl clash).

In [ ]:
mol_butane = Chem.MolFromSmiles("CCCC")
mol_butane = Chem.AddHs(mol_butane)
AllChem.EmbedMolecule(mol_butane, randomSeed=42)

# Find the central C-C-C-C dihedral atoms (0, 1, 2, 3)
conf = mol_butane.GetConformer()
dihedral_atoms = (0, 1, 2, 3)

angles_scan = np.linspace(0, 360, 37)
energies_mmff = []

for ang in angles_scan:
    Chem.rdMolTransforms.SetDihedralDeg(conf, *dihedral_atoms, float(ang))
    # MMFF calculation without modifying dihedral
    prop = AllChem.MMFFGetMoleculeProperties(mol_butane)
    ff = AllChem.MMFFGetMoleculeForceField(mol_butane, prop)
    e = ff.CalcEnergy()
    energies_mmff.append(e)

energies_rel_kj = (np.array(energies_mmff) - min(energies_mmff)) * 4.184  # kcal to kJ

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(angles_scan, energies_rel_kj, "o-", color="#2ca02c", lw=2, markersize=5)
ax.set_title("n-Butane C-C-C-C Torsional Energy Profile (MMFF)", fontsize=12)
ax.set_xlabel("Dihedral Angle $\phi$ (degrees)", fontsize=11)
ax.set_ylabel("Relative Energy (kJ/mol)", fontsize=11)
ax.set_xticks(np.arange(0, 361, 60))
ax.axvline(60, color="gray", ls="--", alpha=0.6, label="Gauche (60°)")
ax.axvline(180, color="blue", ls="--", alpha=0.6, label="Anti (180°)")
ax.axvline(300, color="gray", ls="--", alpha=0.6, label="Gauche (300°)")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

# 3D visualization helper via py3Dmol
try:
    import py3Dmol
    viewer = py3Dmol.view(width=350, height=250)
    viewer.addModel(Chem.MolToMolBlock(mol_butane), "mol")
    viewer.setStyle({"stick": {}})
    viewer.zoomTo()
    print("py3Dmol is available for interactive 3D rendering.")
except ImportError:
    print("py3Dmol not installed in current session.")

## 6. Computational Molecular Editors Overview

Molecular editors fall into two broad categories:

| Tool | Type | Capabilities & Export | License |
|---|---|---|---|
| **Avogadro / Avogadro 2** | 3D Builder & Visualizer | Z-matrix editor, surface electrostatic potentials, optimization | Open-source (GPL) |
| **Schrödinger Maestro** | Professional 3D Suite | Protein-ligand editing, docking preparation, glide grids | Commercial (Free Academic version) |
| **Molden** | 3D Visualizer | Electronic density, molecular orbitals, vibration analysis | Academic free |
| **PyMOL / ChimeraX** | 3D Macromolecular Visualizer | High-resolution ray-tracing, binding site rendering | Open-source / Academic |
| **ChemDraw** | 2D Chemical Sketcher | Publication-grade 2D structures, IUPAC name generation | Commercial |
| **MarvinSketch / JSME** | 2D Web/Desktop Sketchers | SMILES/SMARTS export, RDKit web integration | Free / Open-source |
| **PubChem Sketcher** | 2D Online Sketcher | Instant database cross-referencing and CID retrieval | Public web service |

### Exercise 2.1

Build the 3D coordinates of methane using internal coordinates (Z-matrix) and verify that all 4 C-H bond lengths equal $1.09 \pm 0.01\ \text{Å}$. Place C at origin, H1 along the Z-axis, H2 in the XZ plane with $\angle \text{H}_1-\text{C}-\text{H}_2 = 109.47^\circ$, and H3/H4 using dihedrals $120.0^\circ$ and $240.0^\circ$.

In [ ]:
def build_methane_zmatrix() -> np.ndarray:
    r = 1.09
    alpha_deg = 109.4712
    C = np.array([0.0, 0.0, 0.0])
    H1 = np.array([0.0, 0.0, r])
    
    # H2 in XZ plane
    theta = np.radians(alpha_deg)
    H2 = np.array([r * np.sin(theta), 0.0, r * np.cos(theta)])
    
    # Helper to place H3, H4
    def place_h(phi_deg):
        phi = np.radians(phi_deg)
        # rotated around Z axis relative to H2
        x = r * np.sin(theta) * np.cos(phi)
        y = r * np.sin(theta) * np.sin(phi)
        z = r * np.cos(theta)
        return np.array([x, y, z])
        
    H3 = place_h(120.0)
    H4 = place_h(240.0)
    return np.array([C, H1, H2, H3, H4])

In [ ]:
# Self-check for Exercise 2.1
coords = build_methane_zmatrix()
assert coords.shape == (5, 3)
C_pos = coords[0]
for i in range(1, 5):
    dist = np.linalg.norm(coords[i] - C_pos)
    assert np.isclose(dist, 1.09, atol=0.01)
print('✔ Exercise 2.1 passed!')

### Exercise 2.2

Write a function `rotate_about_x_axis(coords: np.ndarray, angle_deg: float) -> np.ndarray` that rotates an $N \times 3$ coordinate array about the x-axis by `angle_deg` degrees. Rotate your methane coordinates by $30^\circ$ and assert that the distance between C and H1 is conserved.

In [ ]:
def rotate_about_x_axis(coords: np.ndarray, angle_deg: float) -> np.ndarray:
    theta = np.radians(angle_deg)
    Rx = np.array([
        [1.0, 0.0, 0.0],
        [0.0, np.cos(theta), -np.sin(theta)],
        [0.0, np.sin(theta),  np.cos(theta)]
    ])
    return coords @ Rx.T

In [ ]:
# Self-check for Exercise 2.2
m_coords = build_methane_zmatrix()
m_rot = rotate_about_x_axis(m_coords, 30.0)
d_orig = np.linalg.norm(m_coords[1] - m_coords[0])
d_rot = np.linalg.norm(m_rot[1] - m_rot[0])
assert np.isclose(d_orig, d_rot, atol=1e-5)
print('✔ Exercise 2.2 passed!')

### Exercise 2.3

Write a function `compute_dihedral(p0: np.ndarray, p1: np.ndarray, p2: np.ndarray, p3: np.ndarray) -> float` that computes the dihedral angle formed by four sequentially bonded atoms in degrees within the range $[-180, 180]$. Test on the butane anti conformer to verify $\approx 180^\circ$.

In [ ]:
def compute_dihedral(p0: np.ndarray, p1: np.ndarray, p2: np.ndarray, p3: np.ndarray) -> float:
    b0 = -1.0 * (p1 - p0)
    b1 = p2 - p1
    b2 = p3 - p2
    
    # normalise b1
    b1_norm = b1 / np.linalg.norm(b1)
    
    v = b0 - np.dot(b0, b1_norm) * b1_norm
    w = b2 - np.dot(b2, b1_norm) * b1_norm
    
    x = np.dot(v, w)
    y = np.dot(np.cross(b1_norm, v), w)
    return np.degrees(np.arctan2(y, x))

In [ ]:
# Self-check for Exercise 2.3
# Extract coordinates from butane conformer
conf = mol_butane.GetConformer()
Chem.rdMolTransforms.SetDihedralDeg(conf, 0, 1, 2, 3, 180.0)
p_anti = [np.array(conf.GetAtomPosition(i)) for i in range(4)]
dih_anti = compute_dihedral(*p_anti)
assert np.isclose(abs(dih_anti), 180.0, atol=1.0)

Chem.rdMolTransforms.SetDihedralDeg(conf, 0, 1, 2, 3, 60.0)
p_gauche = [np.array(conf.GetAtomPosition(i)) for i in range(4)]
dih_gauche = compute_dihedral(*p_gauche)
assert np.isclose(dih_gauche, 60.0, atol=1.0)
print('✔ Exercise 2.3 passed!')

## Common Pitfalls

1. **Radian vs. Degree Confusion:** Trigonometric functions in numpy (`np.sin`, `np.cos`) expect radians. Passing raw degrees ($109.47$) produces nonsense coordinates.
2. **Dihedral Sign Conventions:** The sign of a dihedral angle depends on the direction of view along the central bond. Following the IUPAC right-hand rule ensures consistency.
3. **Collinear Vectors in NeRF:** If atoms $A, B, C$ are strictly collinear ($180^\circ$ bond angle), the cross product $\vec{u} \times \vec{ab}$ vanishes, leading to division by zero.

## Summary and Key Takeaways

- Methane's 3D coordinates can be derived analytically from $r = 1.09\ \text{Å}$ and $\theta = 54.74^\circ$, producing the textbook $(0.63, 0.89)$ projections.
- Cartesian coordinates are frame-dependent; internal coordinates and distance matrices are frame-invariant.
- A Z-Matrix specifies geometry using bond lengths, bond angles, and torsion angles, eliminating rigid translational and rotational degrees of freedom.
- The NeRF algorithm efficiently translates internal Z-matrices to 3D Cartesian coordinates.
- Conformational scanning of n-butane reveals anti (global minimum at $180^\circ$) and gauche (local minima at $\pm 60^\circ$) energy states.

## References

- Parsons, J. et al. Practical conversion from internal to Cartesian coordinates (NeRF). *J. Comput. Chem.* **2005**, 26, 1063–1068.
- Leach, A. R. *Molecular Modelling: Principles and Applications*. 2nd ed., Pearson, 2001.

In [ ]:
# Self-test check
print("✔ NB02 checks passed")